### 1. Setup & Data gathering

In [ ]:
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import plotly.express as px
import matplotlib.pyplot as plt
import pytrendseries
import math
from scipy import stats
 
 
import warnings
warnings.filterwarnings('ignore')
 
import os
import eikon as ek
import refinitiv.data as rd
 
 
#Scroll through DF
pd.set_option("display.max_rows", None, "display.max_columns", None)
 
 
#Defining Proxy
os.environ['NO_PROXY'] = 'localhost'
os.environ['NO_PROXY'] = '127.0.0.1'
 
 
#Open Session
rd.open_session()
 
#API Key
ek.set_app_key('b7f9e07dd1664cb2b48043ec3321b0e219bc776b')
 
#Todays Datetime
dt_now = datetime.datetime.now()

In [ ]:
# Expanded Global FX Universe (26 Pairs)
pairs = [
    # G10 & Majors
    "EUR=", "JPY=", "GBP=", "CHF=", "AUD=", "NZD=", "CAD=", "SEK=", "NOK=",
    
    # Liquid Emerging Markets
    "MXN=", "ZAR=", "TRY=", "PLN=", "HUF=", "CZK=", "ILS=", "RON=", "SGD=", "THB=",
    
    # Non-Deliverable Emerging Markets (NDFs)
    "BRL=", "CLP=", "KRW=", "TWD=", "INR=", "IDR="
]

# Generate the corresponding 1-Month Forward / NDF RICs
#fwd_pairs = [p.replace("=", "1MV=") for p in pairs]

fwd_pairs = [
    # G10 & Majors
    'EUR1MV=', 'JPY1MV=', 'GBP1MV=', 'CHF1MV=', 'AUD1MV=', 'NZD1MV=', 
    'CAD1MV=', 'SEK1MV=', 'NOK1MV=', 
    
    # Liquid Emerging Markets
    'MXN1MV=', 'ZAR1MV=', 'TRY1MV=', 'PLN1MV=', 'HUF1MV=', 'CZK1MV=', 
    'ILS1MV=', 'RON1MV=', 'SGD1MV=', 'THB1MV=', 
    
    # Non-Deliverable Emerging Markets (NDFs)
    'BRL1MOR=FMD', 'CLP1MOR=FMD', 'KRW1MV=', 'TWD1MV=', 'INR1MV=', 'IDR1MV=']

# Print to verify
print(f"Total Spot Pairs: {len(pairs)}")
print(f"Total Forward Pairs: {len(fwd_pairs)}")

In [ ]:
df_spot = rd.get_history(
    universe=pairs,
    fields=["MID_PRICE"],
    interval="1M",
    start='2005-01-01',
    end='2026-06-30',
    use_field_names_in_headers=True
)

df_fwd = rd.get_history(
    universe=fwd_pairs,
    fields=["MID_PRICE"],
    interval="1M",
    start='2005-01-01',
    end='2026-06-30',
    use_field_names_in_headers=True
)

In [ ]:
# Assuming 'df_spot' and 'df_fwd' are your dataframes, indexed by Date.
# First, let's clean the column names so they are easily matched.
# Spot columns: "EUR=" -> "EUR", "EURCAD=" -> "EURCAD"

df_spot.columns = [c.replace('=', '') for c in df_spot.columns]


# Fwd columns: "EUR1MV=" -> "EUR", "EURCAD1MV=" -> "EURCAD"
#df_fwd.columns = [c.replace('1MV=', '').replace('=', '') for c in df_fwd.columns]

df_fwd.columns = df_fwd.columns.str.replace(r'1M.*$', '', regex=True)

In [ ]:
# Ensure we only calculate on dates where both spot and forward data exist
common_dates = df_spot.index.intersection(df_fwd.index)
df_spot = df_spot.loc[common_dates].sort_index()
df_fwd = df_fwd.loc[common_dates].sort_index()

# Ensure the columns are in the exact same order for vectorized math
df_fwd = df_fwd[df_spot.columns]

In [ ]:
df_cpi = rd.get_history(
    universe=["aUSCPI", "aXZCPI", "aCACPI", "aGBCPI", "aCHCPI", "aJPCPI"],
    fields=["VALUE"] * 6,
    start="1990-01-01",
    end="2026-04-17",
    use_field_names_in_headers=True
)

df_cpi.columns = ['CPI_USD', 'CPI_EUR', 'CPI_CAD', 'CPI_GBP', 'CPI_CHF', 'CPI_JPY']
df_cpi = df_cpi.ffill()